# Guia de Pratica Guiada -- Do Dado ao Modelo
## Como um Coordenador de Data Science do Itau Unibanco ensinaria

**Este notebook e diferente de tudo que voce leu antes.**

Nos guias anteriores voce aprendeu os conceitos.
Aqui voce vai aprender a **pensar como cientista de dados** antes de escrever uma linha de codigo.

---

## Como este guia funciona

Cada secao tem 4 partes obrigatorias:

**[SITUACAO]** A dor de negocio. O que o gestor pediu. O problema real.

**[PENSE ANTES]** Perguntas que voce DEVE responder antes de abrir o Python.
Um cientista de dados senior responde essas perguntas mentalmente em 30 segundos.
Voce vai aprender a fazer o mesmo.

**[CODIGO COMENTADO]** Cada linha explicada. Nao so o "o que", mas o "por que agora".

**[INTERPRETE]** O resultado saiu. O que voce diz para o gestor?
Numero sem interpretacao nao vale nada.

---

## O mapa mental que voce vai internalizar

```
PROBLEMA DE NEGOCIO
        |
        v
[1] QUE TIPO DE PROBLEMA E?
    -> Quero PREVER um numero?          -> REGRESSAO
    -> Quero CLASSIFICAR em categorias? -> CLASSIFICACAO
    -> Quero AGRUPAR sem rotulo?        -> CLUSTERING
    -> Quero REDUZIR dimensoes?         -> PCA
    -> Quero MEDIR IMPACTO de algo?     -> EXPERIMENTACAO
        |
        v
[2] COMO ESTA MEU DADO?
    -> Quantas linhas? Quantas colunas?
    -> Ha valores ausentes? Onde? Por que?
    -> As variaveis sao numericas, categoricas, textuais, temporais?
    -> Ha desbalanceamento no alvo?
    -> Ha vazamento de informacao (data leakage)?
        |
        v
[3] QUAL MODELO COMECAR?
    -> SEMPRE comecar pelo mais simples que resolve o problema
    -> Logistica (classificacao) ou Linear (regressao) como baseline
    -> Depois escalar para XGBoost se necessario
    -> Nunca comecar com deep learning para dados tabulares
        |
        v
[4] COMO AVALIAR?
    -> Qual metrica importa para o NEGOCIO?
    -> Qual o custo de cada tipo de erro?
    -> Como reportar o resultado para quem nao e tecnico?
        |
        v
[5] O QUE FAZER COM O RESULTADO?
    -> Colocar em producao? Como?
    -> Monitorar ao longo do tempo?
    -> Retreinar quando?
```

**Voce vai percorrer esse mapa em cada exercicio.**


---
# Setup

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost --quiet
print('OK!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, math
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, mean_squared_error,
    r2_score, classification_report
)
from xgboost import XGBClassifier

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (12,5), 'axes.grid': True, 'grid.alpha': 0.3})
print('Pronto para praticar!')

---
# CAPITULO 1 -- Explorar Antes de Modelar
## A habilidade mais subestimada do cientista de dados

---

## Por que a exploracao vem antes de tudo?

Imagine que voce e arquiteto. Antes de projetar o edificio,
voce visita o terreno: qual o solo? Ha agua subterranea? Qual o vento?

Modelar sem explorar e construir sem ver o terreno.
Voce pode passar horas num modelo que nunca vai funcionar
porque o dado tem um problema que voce nao viu.

**Regra do coordenador:** para cada hora de modelagem,
gaste pelo menos meia hora explorando o dado antes.

---

## O que voce precisa responder na exploracao

**Sobre o dado em geral:**
- Quantas linhas e colunas? O dado cabe na memoria?
- Qual o periodo? Ha dados historicos suficientes?
- Ha duplicatas? Ha linhas completamente vazias?

**Sobre cada variavel:**
- Qual o tipo? (numerico, categorico, data, texto)
- Ha valores ausentes? Quantos? O ausente e aleatorio ou tem padrao?
- Qual a distribuicao? (normal, log-normal, bimodal, uniforme)
- Ha outliers? Eles sao erros ou sao reais?

**Sobre o alvo (Y):**
- Esta balanceado? (para classificacao: 50/50 vs 99/1)
- Qual a distribuicao? (para regressao: normal vs assimetrico)
- Ha vazamento? (alguma feature so existe depois que Y acontece?)

**Sobre as relacoes entre variaveis:**
- Quais features correlacionam com o alvo?
- Ha multicolinearidade entre features?
- Alguma feature categorica tem muitas categorias? (alta cardinalidade)


In [ ]:
# EXERCICIO 1.1 -- EXPLORACAO COMPLETA DE UM DATASET BANCARIO

# [SITUACAO]
# O gestor de credito PJ trouxe um dataset de clientes e pediu:
# 'Preciso entender quem sao os clientes que inadimpliram.'
# Antes de construir qualquer modelo, voce precisa ENTENDER o dado.

# Gerando o dataset (em producao: seria pd.read_csv('dados_credito_pj.csv'))
np.random.seed(42)
N = 3000

df = pd.DataFrame({
    'cnpj_hash':         [f'CNPJ_{i:05d}' for i in range(N)],
    'faturamento_anual':  np.random.lognormal(13, 1.5, N),
    'tempo_empresa_anos': np.random.exponential(6, N).clip(0.5, 50),
    'n_funcionarios':     np.random.lognormal(2, 1.2, N).clip(1, 5000).astype(int),
    'score_bureau':       np.random.normal(620, 110, N).clip(200, 1000),
    'n_protestos':        np.random.poisson(0.4, N),
    'setor':              np.random.choice(
        ['comercio','industria','servicos','agro','construcao'],
        N, p=[0.35,0.20,0.30,0.10,0.05]
    ),
    'uf':                 np.random.choice(
        ['SP','RJ','MG','RS','PR','outros'], N,
        p=[0.35,0.15,0.15,0.10,0.10,0.15]
    ),
    'tem_garantia':       np.random.binomial(1, 0.40, N),
    'valor_credito':      np.random.lognormal(10, 1.0, N),
})

# Inserindo valores ausentes (como no dado real -- nunca vem limpo!)
idx_miss_score = np.random.choice(N, size=int(N*0.08), replace=False)   # 8% sem score
idx_miss_fat   = np.random.choice(N, size=int(N*0.03), replace=False)   # 3% sem faturamento
df.loc[idx_miss_score, 'score_bureau']      = np.nan
df.loc[idx_miss_fat,   'faturamento_anual'] = np.nan

# Criando o alvo: inadimpliu sim/nao
logit = (
    -2.5
    - 0.003 * df['score_bureau'].fillna(500)
    + 0.8   * df['n_protestos']
    - 0.04  * df['tempo_empresa_anos']
    + 0.4   * (df['setor'] == 'construcao').astype(float)
    - 0.5   * df['tem_garantia']
)
prob = 1 / (1 + np.exp(-logit))
df['inadimpliu'] = np.random.binomial(1, prob)

print('Dataset carregado!')
print(f'Linhas: {len(df):,}  |  Colunas: {df.shape[1]}')

In [ ]:
# PASSO 1: VISAO GERAL -- as primeiras perguntas

# [PENSE ANTES DE EXECUTAR]
# Antes de ver o resultado, responda mentalmente:
# 1. Quantas linhas voce espera? (3.000 -- voce criou)
# 2. Quais colunas sao numericas? Quais sao categoricas?
# 3. Ha alguma coluna que NUNCA deve entrar no modelo? (cnpj_hash -- e um ID!)

print('=== VISAO GERAL DO DATASET ===')
print()
print(df.info())
print()
print('=== PRIMEIRAS 5 LINHAS ===')
print(df.head())
print()

# [INTERPRETE]:
# - object = categorico (precisa de tratamento antes do modelo)
# - float64 = numerico continuo (pode ter NaN)
# - int64 = numerico inteiro
# - cnpj_hash: ID -- NUNCA entra no modelo (identificador, nao feature)
print()
print('COLUNAS E SEUS TIPOS:')
for col in df.columns:
    tipo = df[col].dtype
    nuniq = df[col].nunique()
    print(f'  {col:<25} {str(tipo):<12} {nuniq} valores unicos')
print()
print('[!] REGRA: ID e chave primaria NUNCA entra no modelo.')
print('    Se entrar: o modelo vai memorizar o ID e nao vai generalizar.')

In [ ]:
# PASSO 2: VALORES AUSENTES -- mapa completo

# [PENSE ANTES]
# Antes de ver, responda:
# 1. Quais colunas tem valores ausentes?
# 2. O ausente e aleatorio (MAR) ou tem padrao (MNAR)?
# 3. O que fazer com cada tipo?

print('=== ANALISE DE VALORES AUSENTES ===')
print()

ausentes = df.isnull().sum()
pct_ausentes = df.isnull().mean() * 100

df_miss = pd.DataFrame({
    'N ausentes': ausentes,
    '% ausentes': pct_ausentes.round(2)
}).query('`N ausentes` > 0')

if len(df_miss) == 0:
    print('Nenhum valor ausente! (raro na vida real)')
else:
    print(df_miss)
    print()

    # INVESTIGANDO O PADRAO DOS AUSENTES
    # Pergunta: clientes SEM score_bureau tem taxa de inadimplencia diferente?
    # Se sim: o ausente NAO e aleatorio -- carrega informacao!
    print('Taxa de inadimplencia por ausencia de score_bureau:')
    df['sem_score'] = df['score_bureau'].isnull().astype(int)
    print(df.groupby('sem_score')['inadimpliu'].mean().rename(
        {0: 'TEM score', 1: 'SEM score'}
    ).to_string())
    print()
    print('[!] SE a taxa e diferente: o ausente carrega informacao.')
    print('    TRATAMENTO: criar indicador binario "sem_score" E imputar o valor.')
    print('    NAO simplesmente dropar a linha -- voce perde informacao.')
    print()
    print('ESTRATEGIAS DE IMPUTACAO:')
    estrategias = [
        ('Mediana',    'Numerica continua, outliers presentes -- RECOMENDADO'),
        ('Media',      'Numerica continua, sem outliers, distribuicao simetrica'),
        ('Moda',       'Categorica -- valor mais frequente'),
        ('Modelo KNN', 'Quando o ausente depende de outras variaveis'),
        ('Indicador',  'SEMPRE criar coluna binaria "era_ausente" junto com a imputacao'),
    ]
    for nome, quando in estrategias:
        print(f'  {nome:<15}: {quando}')

In [ ]:
# PASSO 3: DISTRIBUICOES -- vendo a forma dos dados

# [PENSE ANTES]
# Para cada variavel numerica, pergunte:
# 1. A distribuicao e simetrica ou assimetrica?
# 2. Ha outliers extremos? (pontos muito longe da maioria)
# 3. Faz sentido de negocio? (faturamento negativo seria erro)

numericas = ['faturamento_anual', 'tempo_empresa_anos', 'score_bureau', 'valor_credito']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(numericas):
    dados = df[col].dropna()

    # Histograma
    axes[0, i].hist(dados, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, i].axvline(dados.mean(),   color='red',    linestyle='--', lw=2, label=f'Media={dados.mean():.0f}')
    axes[0, i].axvline(dados.median(), color='orange', linestyle='--', lw=2, label=f'Mediana={dados.median():.0f}')
    axes[0, i].set_title(f'{col}\nAssimetria={dados.skew():.2f}')
    axes[0, i].legend(fontsize=7)

    # Boxplot (detecta outliers)
    axes[1, i].boxplot(dados, vert=True)
    axes[1, i].set_title(f'{col}\nBoxplot')

plt.suptitle('Distribuicoes das Variaveis Numericas', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('COMO INTERPRETAR:')
print('  Assimetria proximo de 0: distribuicao simetrica (como Normal)')
print('  Assimetria > 1 ou < -1: distribuicao assimetrica (considere log)')
print('  Boxplot: pontos alem dos "bigodes" sao outliers')
print()
print('DECISAO DE TRATAMENTO:')
decisoes = [
    ('faturamento_anual', 'Log-normal (assimetrica) -> aplicar log1p antes do modelo'),
    ('score_bureau',      'Aproximadamente normal -> usar direto ou z-score'),
    ('valor_credito',     'Log-normal -> aplicar log1p antes do modelo'),
    ('tempo_empresa_anos','Exponencial -> considerar log ou usar direto'),
]
for col, decisao in decisoes:
    print(f'  {col:<25}: {decisao}')

In [ ]:
# PASSO 4: ANALISE DO ALVO -- o que voce vai prever

# [PENSE ANTES]
# 1. O alvo e binario? Multiclasse? Continuo?
# 2. Ha desbalanceamento? (menos de 20% de um lado = desbalanceado)
# 3. Se desbalanceado: acuracia e inutil -- qual metrica usar?

print('=== ANALISE DO ALVO: inadimpliu ===')
print()
contagem = df['inadimpliu'].value_counts()
pct      = df['inadimpliu'].value_counts(normalize=True) * 100

print(f'  0 (adimplente):  {contagem[0]:,} clientes ({pct[0]:.1f}%)')
print(f'  1 (inadimplente): {contagem[1]:,} clientes ({pct[1]:.1f}%)')
print()

taxa = df['inadimpliu'].mean()
if taxa < 0.10 or taxa > 0.90:
    print(f'[!] BASE DESBALANCEADA: {taxa*100:.1f}% de positivos')
    print('    IMPLICACAO: acuracia e inutil como metrica!')
    print(f'    Modelo idiota (preve sempre 0): acuracia = {(1-taxa)*100:.1f}%')
    print('    USE: AUC-ROC, Precision, Recall, F1')
elif taxa < 0.30 or taxa > 0.70:
    print(f'[!] BASE MODERADAMENTE DESBALANCEADA: {taxa*100:.1f}%')
    print('    Use AUC-ROC como metrica principal')
else:
    print(f'Base relativamente balanceada: {taxa*100:.1f}%')
    print('Acuracia pode ser usada, mas AUC-ROC e mais informativa')

print()
print('TAXA DE INADIMPLENCIA POR SEGMENTO (analise descritiva):')
print()
for col in ['setor', 'uf', 'tem_garantia']:
    print(f'  Por {col}:')
    tab = df.groupby(col)['inadimpliu'].agg(['mean','count']).sort_values('mean', ascending=False)
    tab.columns = ['taxa_inadim', 'n_clientes']
    tab['taxa_inadim'] = (tab['taxa_inadim']*100).round(1)
    print(tab.to_string())
    print()

print('[->] ESTA ANALISE DESCRITIVA ja orienta as features mais relevantes!')
print('     Setor com maior inadimplencia provavelmente sera feature importante.')

In [ ]:
# PASSO 5: CORRELACAO COM O ALVO -- quais features prometem

# [PENSE ANTES]
# Para cada feature numerica, calcule a correlacao com o alvo.
# Feature com correlacao alta (positiva ou negativa) = mais informativa.
# Mas cuidado: correlacao nao e causalidade!

print('=== CORRELACAO DAS FEATURES COM O ALVO ===')
print()

df_corr = df[['faturamento_anual','tempo_empresa_anos','n_funcionarios',
              'score_bureau','n_protestos','tem_garantia',
              'valor_credito','inadimpliu']].copy()

correlacoes = df_corr.corr()['inadimpliu'].drop('inadimpliu').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['coral' if v > 0 else 'steelblue' for v in correlacoes]
correlacoes.plot(kind='barh', ax=ax, color=cores)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlacao de Pearson com o Alvo (inadimpliu)\n'
             'Vermelho = aumenta risco | Azul = reduz risco')
ax.set_xlabel('Correlacao')
plt.tight_layout()
plt.show()

print('Correlacoes ordenadas (mais forte primeiro):')
for feat, corr in correlacoes.abs().sort_values(ascending=False).items():
    direcao = '+' if correlacoes[feat] > 0 else '-'
    print(f'  {feat:<25}: r={direcao}{corr:.3f}')
print()
print('[!] CUIDADO COM CORRELACAO PROXIMA DE 1.0:')
print('    Pode indicar DATA LEAKAGE -- feature que so existe DEPOIS do evento.')
print('    Ex: "valor_em_atraso" correlacionada com inadimplencia = leakage!')

---
# CAPITULO 2 -- Preparacao dos Dados
## O trabalho invisivel que determina o sucesso do modelo

---

## A regra de ouro

**Garbage In, Garbage Out.**

O melhor algoritmo do mundo nao compensa dados mal preparados.
Inversamente: dados bem preparados podem fazer um modelo simples funcionar muito bem.

Preparacao e onde os cientistas de dados passam 60-70% do tempo.

---

## O que precisa ser feito (em ordem)

**1. Separar treino e teste PRIMEIRO**
Antes de qualquer preprocessamento.
Se voce preprocessar com o dado todo, vai ter data leakage.

**2. Tratar valores ausentes**
Imputar com mediana/moda (do treino, aplicar no teste).
Criar indicador binario "era_ausente".

**3. Tratar outliers**
Decidir: sao erros? sao reais? Capping (limitar ao percentil 99)?

**4. Codificar variaveis categoricas**
One-hot encoding para categorias sem ordem.
Label encoding para categorias com ordem (baixo < medio < alto).

**5. Escalar variaveis numericas (se necessario)**
Obrigatorio para: SVM, Logistica com regularizacao, KNN, PCA.
Nao necessario para: arvores, Random Forest, XGBoost.

**6. Engenharia de features**
Criar novas variaveis que capturam padroes que o modelo sozinho nao capturaria.

---

## Sobre o Pipeline do sklearn

Voce SEMPRE deve usar Pipeline para encadear preprocessamento + modelo.
Por que? Evita data leakage automaticamente.
O Pipeline re-fita o preprocessamento apenas nos dados de treino de cada fold.


In [ ]:
# EXERCICIO 2.1 -- PREPARACAO COMPLETA DO DADO

# [PENSE ANTES]
# 1. Qual coluna e o ID? Deve ser removida.
# 2. Qual coluna e o alvo? Deve ser separada.
# 3. Quais colunas sao numericas? Quais sao categoricas?
# 4. Ha ausentes? Como tratar?
# 5. Ha categoricas? Como codificar?

# PASSO 1: Definir quais colunas sao o que
COLUNA_ID     = 'cnpj_hash'        # identificador -- FORA do modelo
COLUNA_ALVO   = 'inadimpliu'       # o que queremos prever
COLUNAS_DROP  = [COLUNA_ID, COLUNA_ALVO, 'sem_score']  # nao entram

# PASSO 2: Engenharia de features ANTES do split
# (features derivadas de logica de negocio, nao de estatistica)
df_prep = df.copy()

# Feature: log do faturamento (normaliza a distribuicao lognormal)
df_prep['log_faturamento'] = np.log1p(df_prep['faturamento_anual'])

# Feature: log do valor do credito
df_prep['log_valor_credito'] = np.log1p(df_prep['valor_credito'])

# Feature: ratio valor credito / faturamento (alavancagem)
df_prep['ratio_credito_fat'] = (
    df_prep['valor_credito'] / df_prep['faturamento_anual'].fillna(1)
).clip(0, 100)

# Feature: indicador de ausencia de score (o ausente carrega info!)
df_prep['sem_score_bureau'] = df_prep['score_bureau'].isnull().astype(int)

# PASSO 3: Separar features e alvo
features_usar = [c for c in df_prep.columns if c not in COLUNAS_DROP]
X = df_prep[features_usar]
y = df_prep[COLUNA_ALVO]

print('Features que entram no modelo:')
for f in features_usar:
    tipo = X[f].dtype
    ausentes = X[f].isnull().sum()
    print(f'  {f:<30} {str(tipo):<12} ausentes: {ausentes}')
print(f'\nTotal: {len(features_usar)} features, {len(X):,} observacoes')

In [ ]:
# PASSO 4: SPLIT TREINO/TESTE

# [PENSE ANTES]
# 1. Qual proporcao usar? (80/20 e padrao para N > 1000)
# 2. Precisa de stratify? (SIM -- alvo binario com desbalanceamento)
# 3. Devo usar split aleatorio ou temporal?
#    TEMPORAL: dados de clientes com timestamp -> split por data
#    ALEATORIO: dados cross-section -> split aleatorio com stratify

# Identificando colunas numericas e categoricas para preprocessamento
CATS = ['setor', 'uf']                  # colunas categoricas
NUMS = [c for c in features_usar if c not in CATS]  # colunas numericas

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.20,     # 20% para teste
    random_state=42,    # reproducibilidade
    stratify=y          # garante mesma proporcao de inadimplentes em tr e te
)

print(f'Treino: {len(X_tr):,} observacoes  |  Teste: {len(X_te):,} observacoes')
print(f'Taxa inadimplencia no treino: {y_tr.mean()*100:.1f}%')
print(f'Taxa inadimplencia no teste:  {y_te.mean()*100:.1f}%')
print()
print('[!] As taxas devem ser proximas (garantido pelo stratify).')
print('    Se nao fossem: o modelo treina em distribuicao diferente do teste.')
print()
print('CATEGORICAS:', CATS)
print('NUMERICAS:', NUMS[:5], '... (e mais', len(NUMS)-5, ')')

In [ ]:
# PASSO 5: PREPROCESSAMENTO COM PIPELINE
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# [PENSE ANTES]
# Para cada tipo de coluna, qual tratamento?
#
# NUMERICAS:
#   -> Imputar ausentes com MEDIANA (robusta a outliers)
#   -> Escalar com StandardScaler (z-score)
#   -> Por que mediana e nao media? Porque dados financeiros tem outliers.
#
# CATEGORICAS:
#   -> Imputar ausentes com 'desconhecido' (moda)
#   -> OneHotEncoding: cria coluna binaria para cada categoria
#   -> handle_unknown='ignore': ignora categoria nova no teste

# Pipeline para numericas: imputar -> escalar
pipe_num = Pipeline([
    ('imput', SimpleImputer(strategy='median')),   # preenche NaN com mediana do treino
    ('scale', StandardScaler())                    # z-score: (x-media)/std
])

# Pipeline para categoricas: imputar -> one-hot
pipe_cat = Pipeline([
    ('imput', SimpleImputer(strategy='most_frequent')),  # preenche NaN com moda
    ('ohe',   OneHotEncoder(handle_unknown='ignore',     # ignora cat. nova no teste
                            sparse_output=False))        # retorna array denso
])

# ColumnTransformer: aplica pipe_num nas numericas e pipe_cat nas categoricas
preprocessador = ColumnTransformer([
    ('num', pipe_num, NUMS),  # aplica pipe_num nas colunas NUMS
    ('cat', pipe_cat, CATS),  # aplica pipe_cat nas colunas CATS
])

# Testando o preprocessador
X_tr_proc = preprocessador.fit_transform(X_tr)  # FIT apenas no treino!
X_te_proc = preprocessador.transform(X_te)      # apenas transforma o teste

print(f'Shape antes do preprocessamento: {X_tr.shape}')
print(f'Shape depois (com OHE):          {X_tr_proc.shape}')
print()
print('Por que o numero de colunas aumentou?')
print('  OneHotEncoding criou uma coluna por categoria.')
print(f'  setor tem {X["setor"].nunique()} categorias -> {X["setor"].nunique()} colunas novas')
print(f'  uf tem {X["uf"].nunique()} categorias -> {X["uf"].nunique()} colunas novas')
print()
print('[!] REGRA: fit_transform() APENAS no treino.')
print('    transform() no teste (sem re-fitar).')
print('    Caso contrario: data leakage -- o preprocessador viu o teste!')

---
# CAPITULO 3 -- Construindo e Avaliando Modelos
## A hierarquia de complexidade que todo cientista segue

---

## A regra dos modelos

**Comece sempre pelo modelo mais simples.**

Nao porque os modelos simples sao melhores.
Mas porque eles te dao um baseline: se o XGBoost nao e melhor
que a logistica, algo esta errado com o dado ou com o XGBoost.

**Hierarquia recomendada:**

```
1. Modelo Idiota (baseline cego)
   -> Preve sempre a classe majoritaria
   -> Acuracia = taxa da classe majoritaria
   -> SE seu modelo nao bate isso: algo esta muito errado

2. Logistica (baseline inteligente)
   -> Simples, interpretavel, calibrado
   -> Se logistica ja resolve: nao complique

3. Random Forest
   -> Captura nao-linearidades
   -> Menos senssivel a hiperparametros que XGBoost

4. XGBoost
   -> Estado da arte para dados tabulares
   -> Requer mais tuning

5. (raramente) Redes Neurais
   -> So se os dados sao muito grandes e os outros ja foram tentados
```

---

## Cross-Validation: a unica forma honesta de avaliar

NUNCA avalie o modelo no dado de treino.
NUNCA avalie no teste ate estar pronto para reportar o resultado final.

Use Cross-Validation no conjunto de treino para:
- Comparar modelos (qual e melhor?)
- Tunar hiperparametros (qual alpha? qual depth?)
- Estimar a performance real (que AUC posso esperar em producao?)

O conjunto de teste e sagrado: use apenas UMA VEZ, no final.


In [ ]:
# EXERCICIO 3.1 -- CONSTRUINDO A HIERARQUIA DE MODELOS

# [PENSE ANTES]
# 1. Qual e o tipo de problema? (classificacao binaria)
# 2. O dado e desbalanceado? (sim -- usar AUC, nao acuracia)
# 3. Qual metrica de CV usar? (roc_auc -- discriminacao entre classes)
# 4. Qual tipo de CV usar? (KFold(5, shuffle=False) -- padrao da prova Itau)

kf = KFold(n_splits=5, shuffle=False)  # padrao confirmado da prova

# MODELO 0: Idiota (baseline cego)
# Preve sempre a classe majoritaria (0 = adimplente)
taxa_0 = 1 - y_tr.mean()  # proporcao da classe 0

# MODELOS PARA COMPARAR
modelos = {
    'Logistica (C=1.0)': Pipeline([
        ('pre', preprocessador),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
    ]),
    'Logistica (C=0.1)': Pipeline([
        ('pre', preprocessador),
        ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessador),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=6,
                                       random_state=42))
    ]),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        reg_lambda=1.0, verbosity=0, random_state=42
    ),
}

# NOTA: XGBoost recebe dado preprocessado separadamente
# (nao suporta Pipeline com ColumnTransformer da mesma forma)

print('=== COMPARACAO DE MODELOS (5-fold CV) ===')
print()
print(f'Baseline idiota (preve sempre 0): AUC = 0.500 (definicao)')
print(f'Acuracia idiota:                  {taxa_0*100:.1f}%')
print()
print(f'{"Modelo":<25} {"AUC Treino":>12} {"AUC Val":>10} {"Std":>7} {"Gap":>8}')
print('-' * 70)

resultados = {}

for nome, modelo in modelos.items():
    if nome == 'XGBoost':
        # XGBoost usa dado preprocessado
        res = cross_validate(
            modelo, X_tr_proc, y_tr,
            cv=kf, scoring='roc_auc', return_train_score=True
        )
    else:
        res = cross_validate(
            modelo, X_tr, y_tr,
            cv=kf, scoring='roc_auc', return_train_score=True
        )

    tr_m  = res['train_score'].mean()
    va_m  = res['test_score'].mean()
    va_s  = res['test_score'].std()
    gap   = tr_m - va_m
    resultados[nome] = {'auc_tr': tr_m, 'auc_val': va_m, 'std': va_s}

    print(f'{nome:<25} {tr_m:>12.4f} {va_m:>10.4f} {va_s:>7.4f} {gap:>8.4f}')

print()
melhor = max(resultados, key=lambda k: resultados[k]['auc_val'])
print(f'Melhor modelo no CV: {melhor}')
print(f'AUC de validacao: {resultados[melhor]["auc_val"]:.4f} +/- {resultados[melhor]["std"]:.4f}')

In [ ]:
# EXERCICIO 3.2 -- INTERPRETANDO OS RESULTADOS DO CV

# [PENSE ANTES]
# Olhando a tabela acima, responda:
# 1. Qual modelo tem o maior AUC de validacao? (melhor generalizacao)
# 2. Algum modelo tem gap grande entre treino e validacao? (overfit)
# 3. Qual modelo tem menor desvio padrao? (mais estavel)
# 4. Vale a pena a complexidade extra do XGBoost vs Logistica?

print('=== DIAGNOSTICO DOS MODELOS ===')
print()

for nome, res in resultados.items():
    gap = res['auc_tr'] - res['auc_val']
    if gap > 0.05:
        diag = 'ATENCAO: gap grande -> possivel overfitting'
    elif res['auc_val'] < 0.60:
        diag = 'ATENCAO: AUC baixo -> modelo nao discrimina bem'
    elif res['std'] > 0.03:
        diag = 'ATENCAO: alta variancia -> instavel entre folds'
    else:
        diag = 'OK'
    print(f'  {nome:<25}: {diag}')

print()
print('PRINCIPIO DA PARCIMONIA (Occam Razor):')
print('  Se a Logistica tem AUC=0.82 e o XGBoost tem AUC=0.83,')
print('  escolha a Logistica -- ganho de 1pp nao justifica a complexidade extra.')
print()
print('QUANDO ESCOLHER O MODELO MAIS COMPLEXO:')
print('  - AUC significativamente maior (> 2-3pp de diferenca)')
print('  - O problema tem relacoes altamente nao-lineares')
print('  - Voce tem dados suficientes para suportar a complexidade')
print('  - A interpretabilidade NAO e requisito (logista exige, prova nao)')

In [ ]:
# EXERCICIO 3.3 -- AVALIACAO FINAL NO CONJUNTO DE TESTE

# [PENSE ANTES]
# VOCE SO PODE FAZER ISSO UMA VEZ.
# Se voce olhar o resultado e ajustar o modelo, voce contamina o teste.
# O teste deve simular o desempenho em producao -- dados que o modelo nunca viu.

# Treinando o melhor modelo no treino completo
# (usando o pipeline completo para nao ter leakage)
modelo_final = Pipeline([
    ('pre', preprocessador),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])
modelo_final.fit(X_tr, y_tr)

# Avaliando no teste (UMA UNICA VEZ)
prob_te   = modelo_final.predict_proba(X_te)[:,1]
y_pred_te = (prob_te >= 0.5).astype(int)
auc_te    = roc_auc_score(y_te, prob_te)

from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_te, prob_te)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Curva ROC
axes[0].plot(fpr, tpr, 'b-', lw=2.5, label=f'Modelo (AUC={auc_te:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1.5, label='Aleatorio (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC -- Conjunto de Teste'); axes[0].legend()

# Matriz de confusao
cm = confusion_matrix(y_te, y_pred_te)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Previsto 0','Previsto 1'],
            yticklabels=['Real 0','Real 1'])
axes[1].set_title('Matriz de Confusao\n(threshold=0.5)')

# Distribuicao dos scores
axes[2].hist(prob_te[y_te==0], bins=30, alpha=0.6, color='steelblue',
             label='Adimplente', density=True)
axes[2].hist(prob_te[y_te==1], bins=30, alpha=0.6, color='coral',
             label='Inadimplente', density=True)
axes[2].set_xlabel('P(inadimplencia)'); axes[2].set_ylabel('Densidade')
axes[2].set_title('Separacao dos Scores'); axes[2].legend()
plt.tight_layout(); plt.show()

print('=== RESULTADO FINAL NO TESTE ===')
print(classification_report(y_te, y_pred_te, target_names=['Adimplente','Inadimplente']))
print(f'AUC ROC: {auc_te:.4f}')
print()
print('COMO INTERPRETAR PARA O GESTOR:')
print(f'  "O modelo separa adimplentes de inadimplentes com {auc_te*100:.0f}% de acuracia."')
print(f'  "Se ordenarmos 1000 clientes por score, os 100 primeiros terao {tpr[int(len(tpr)*0.1)]*100:.0f}% dos inadimplentes."')

---
# CAPITULO 4 -- Regressao na Pratica
## Quando o alvo e um numero, nao uma categoria

---

## Quando usar regressao

- Quero prever o **faturamento** de um cliente no proximo mes
- Quero prever o **valor** que um cliente vai transacionar
- Quero prever o **NPS** medio de um segmento
- Quero prever o **tempo** ate a proxima inadimplencia

A diferenca fundamental:
- Classificacao: Y e categorico (0 ou 1, A ou B ou C)
- Regressao: Y e numerico continuo (qualquer valor real)

---

## Armadilhas especificas de regressao

**1. MAPE com zeros:** se Y pode ser zero, NUNCA use MAPE.

**2. R^2 nunca diminui:** ao adicionar mais variaveis, R^2 nunca cai.
Use R^2 ajustado para comparar modelos com diferente numero de features.

**3. Distribuicao do alvo:** se Y e muito assimetrico (lognormal),
considere prever log(Y) e depois transformar de volta.

**4. Heterocedasticidade:** a variancia dos erros pode crescer com Y.
Isso viola premissas do OLS e exige correcao.


In [ ]:
# EXERCICIO 4.1 -- REGRESSAO: PREVENDO VOLUME DE TRANSACOES

# [SITUACAO]
# O gestor de produtos quer saber: qual o volume de transacoes
# que cada cliente PJ vai gerar no proximo mes?
# Isso permite alocar recursos de atendimento e planejar a capacidade.

# [PENSE ANTES]
# 1. O alvo (volume) e continuo? Sim -> REGRESSAO
# 2. A distribuicao do alvo e simetrica ou assimetrica?
#    Volume e tipicamente lognormal (assimetrico a direita)
#    -> Considerar prever log(volume) e depois exp(previsao)
# 3. Qual metrica usar?
#    -> RMSE (mesma unidade que o alvo, intuitivo)
#    -> MAE (mais robusto a outliers)
#    -> R^2 (proporcao da variancia explicada)
#    -> MAPE APENAS SE ZERO NAO E POSSIVEL

np.random.seed(42)
N4 = 2000

df4 = pd.DataFrame({
    'n_produtos':     np.random.poisson(3, N4) + 1,
    'tempo_conta':    np.random.exponential(4, N4).clip(0.1, 30),
    'n_func':         np.random.lognormal(2, 1, N4).clip(1, 1000).astype(int),
    'score_pj':       np.random.normal(650, 80, N4).clip(300, 900),
    'setor':          np.random.choice(['comercio','servicos','industria'], N4,
                                       p=[0.4, 0.4, 0.2]),
})

# Volume = lognormal (tipico de dados financeiros)
mu_vol = (
    8.0
    + 0.2  * df4['n_produtos']
    + 0.05 * df4['tempo_conta']
    + 0.003* df4['score_pj']
    + 0.5  * np.log1p(df4['n_func'])
)
df4['volume_mensal'] = np.random.lognormal(mu_vol, sigma=0.8)

print('=== ANALISE DO ALVO: volume_mensal ===')
print(f'  Media:    R${df4["volume_mensal"].mean():>12,.0f}')
print(f'  Mediana:  R${df4["volume_mensal"].median():>12,.0f}')
print(f'  Assimetria: {df4["volume_mensal"].skew():.2f}  (> 1 = muito assimetrico)')
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df4['volume_mensal'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Volume bruto\nassimetrico a direita (lognormal)')

axes[1].hist(np.log1p(df4['volume_mensal']), bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('log(Volume)\naproximadamente normal')

axes[2].boxplot(df4['volume_mensal'])
axes[2].set_title('Boxplot\nOutliers na cauda direita')
plt.tight_layout(); plt.show()

print('[!] CONCLUSAO: prever log(volume) e mais adequado.')
print('    A relacao log-linear e mais bem comportada para o modelo.')
print('    Depois: exp(previsao) para voltar a escala original.')

In [ ]:
# EXERCICIO 4.2 -- PIPELINE DE REGRESSAO COMPLETO
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Preparando o alvo em escala log
df4['log_volume'] = np.log1p(df4['volume_mensal'])

features4 = ['n_produtos', 'tempo_conta', 'n_func', 'score_pj', 'setor']
X4 = df4[features4]
y4 = df4['log_volume']  # modelamos o LOG do volume

X4_tr, X4_te, y4_tr, y4_te = train_test_split(
    X4, y4, test_size=0.2, random_state=42  # sem stratify para regressao
)

NUMS4 = ['n_produtos', 'tempo_conta', 'n_func', 'score_pj']
CATS4 = ['setor']

pre4 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), NUMS4),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore',
                                            sparse_output=False))]), CATS4)
])

# Modelos de regressao para comparar
modelos4 = {
    'Regressao Linear': Pipeline([('pre', pre4), ('reg', LinearRegression())]),
    'Ridge (alpha=1)':  Pipeline([('pre', pre4), ('reg', Ridge(alpha=1.0))]),
    'Ridge (alpha=10)': Pipeline([('pre', pre4), ('reg', Ridge(alpha=10.0))]),
}

kf4 = KFold(n_splits=5, shuffle=False)

print('=== COMPARACAO DE MODELOS DE REGRESSAO (5-fold CV) ===')
print()
print(f'{"Modelo":<25} {"RMSE treino":>12} {"RMSE val":>10} {"R2 val":>8}')
print('-' * 60)

for nome, modelo in modelos4.items():
    res_mse = cross_validate(modelo, X4_tr, y4_tr, cv=kf4,
                              scoring='neg_mean_squared_error',
                              return_train_score=True)
    res_r2  = cross_validate(modelo, X4_tr, y4_tr, cv=kf4,
                              scoring='r2')

    rmse_tr = np.sqrt(-res_mse['train_score'].mean())
    rmse_va = np.sqrt(-res_mse['test_score'].mean())
    r2_va   = res_r2['test_score'].mean()

    print(f'{nome:<25} {rmse_tr:>12.4f} {rmse_va:>10.4f} {r2_va:>8.4f}')

print()
print('LEMBRE: estas metricas estao na escala do LOG do volume.')
print('Para interpretar em R$: exp(previsao) - 1')
print()

# Avaliacao final
melhor4 = Pipeline([('pre', pre4), ('reg', LinearRegression())])
melhor4.fit(X4_tr, y4_tr)

log_prev = melhor4.predict(X4_te)
vol_prev = np.expm1(log_prev)    # expm1 = exp(x) - 1, inverso de log1p
vol_real = np.expm1(y4_te)

rmse_reais = np.sqrt(mean_squared_error(vol_real, vol_prev))
mae_reais  = np.mean(np.abs(vol_real - vol_prev))
r2_reais   = r2_score(vol_real, vol_prev)

print(f'=== RESULTADO FINAL NA ESCALA ORIGINAL (R$) ===')
print(f'  RMSE: R${rmse_reais:,.0f}  (erro medio quadratico -- penaliza erros grandes)')
print(f'  MAE:  R${mae_reais:,.0f}   (erro medio absoluto -- mais intuitivo)')
print(f'  R^2:  {r2_reais:.4f}  (explica {r2_reais*100:.0f}% da variancia do volume)')
print()
print('PARA O GESTOR:')
print(f'  "O modelo preve o volume mensal com erro medio de R${mae_reais:,.0f}."')
print(f'  "Ele explica {r2_reais*100:.0f}% da variacao de volume entre clientes."')

---
# CAPITULO 5 -- O Ciclo Completo de um Projeto Real
## Do problema ao insight, do insight a acao

---

## O que falta depois do modelo

A maioria dos tutoriais para quando o modelo esta treinado.
A realidade: o modelo pronto e apenas 30% do trabalho.

**O ciclo completo:**

```
1. ENTENDER O PROBLEMA     (reuniao com o negocio -- o que voce quer?)
2. OBTER O DADO            (SQL, APIs, CSV, data warehouse)
3. EXPLORAR O DADO         (analise exploratoria -- o que o dado diz?)
4. PREPARAR O DADO         (limpeza, features, encoding, split)
5. MODELAR                 (baseline -> complexidade crescente)
6. AVALIAR                 (CV, metricas de negocio, interpretacao)
7. COMUNICAR               (traduzir AUC em R$ e decisoes)
8. COLOCAR EM PRODUCAO     (pipeline, endpoint, batch)
9. MONITORAR               (PSI, drift, retreino)
10. ITERAR                 (o modelo degrada -- recomecar)
```

---

## Como escrever SQL para obter os dados

Em 90% dos casos, voce vai trabalhar com dados em um banco de dados.
Saber SQL bem e pre-requisito para Data Science em empresas.

**Pattern tipico de query para um dataset de propensao:**

```sql
WITH base_clientes AS (
    SELECT
        c.cnpj,
        c.cnae,
        c.porte,
        c.score_pj,
        COUNT(t.id_transacao) AS n_transacoes_90d,
        AVG(t.valor)          AS vol_medio_90d,
        MAX(t.data)           AS ultima_transacao
    FROM clientes c
    LEFT JOIN transacoes t
        ON c.cnpj = t.cnpj
        AND t.data >= DATEADD(day, -90, GETDATE())
    GROUP BY c.cnpj, c.cnae, c.porte, c.score_pj
),
base_alvo AS (
    SELECT cnpj,
           MAX(CASE WHEN aceitou_oferta = 1 THEN 1 ELSE 0 END) AS aceitou
    FROM ofertas
    WHERE data_oferta BETWEEN '2024-01-01' AND '2024-06-30'
    GROUP BY cnpj
)
SELECT
    c.*,
    COALESCE(a.aceitou, 0) AS aceitou
FROM base_clientes c
LEFT JOIN base_alvo a ON c.cnpj = a.cnpj
WHERE c.cnpj IS NOT NULL
```

**Regras de ouro do SQL para ML:**
1. Nunca use informacao do futuro (data de referencia explicita)
2. LEFT JOIN para manter todos os clientes, mesmo sem transacao
3. COALESCE para tratar nulos (nao aceitou = 0, nao ausente)
4. Testar o SQL em um subset pequeno antes de rodar na base toda


In [ ]:
# EXERCICIO 5.1 -- CICLO COMPLETO: PROPENSAO A PRODUTO

# [SITUACAO COMPLETA]
# O gestor de produtos PJ trouxe o seguinte pedido:
# 'Quero saber quem na minha carteira tem maior probabilidade
# de contratar o produto Antecipacao de Recebiveis no proximo mes.
# Tenho R$20/cliente de budget de campanha e quero maximizar contratos.'

# [MAPA MENTAL -- responda antes de codar]
# Tipo de problema? CLASSIFICACAO BINARIA (aceitou / nao aceitou)
# Alvo balanceado? Propocao tipica de aceite = 5-15% -> DESBALANCEADO
# Metrica de negocio? LIFT (quanto melhor que aleatorio?)
# Metrica tecnica? AUC-ROC
# Modelo inicial? Logistica como baseline
# O que entrega ao gestor? Lista ordenada por propensao + impacto financeiro

np.random.seed(42)
N5 = 5000

# Dados dos clientes (como viriam de um SQL)
df5 = pd.DataFrame({
    'cnpj':             [f'CNPJ_{i:05d}' for i in range(N5)],
    'tempo_conta':      np.random.exponential(4, N5).clip(0.1, 30),
    'vol_mensal':       np.random.lognormal(7, 1.5, N5),
    'n_produtos':       np.random.poisson(3, N5) + 1,
    'score_pj':         np.random.normal(650, 80, N5).clip(300, 900),
    'ja_tem_credito':   np.random.binomial(1, 0.35, N5),
    'inadim_hist':      np.random.binomial(1, 0.08, N5),
    'setor':            np.random.choice(
        ['comercio','servicos','industria','agro'],
        N5, p=[0.40, 0.35, 0.20, 0.05]
    ),
})

# Alvo: aceitou a oferta de antecipacao
logit5 = (
    -3.0
    + 0.1  * df5['tempo_conta']
    + 0.3  * np.log1p(df5['vol_mensal'])
    + 0.2  * df5['n_produtos']
    + 0.005* (df5['score_pj'] - 650)
    + 0.5  * df5['ja_tem_credito']
    - 2.0  * df5['inadim_hist']
    + 0.3  * (df5['setor'] == 'comercio').astype(float)
)
prob5 = 1 / (1 + np.exp(-logit5))
df5['aceitou'] = np.random.binomial(1, prob5)

taxa5 = df5['aceitou'].mean()
print(f'Taxa de aceite: {taxa5*100:.1f}%  (base desbalanceada!)')
print(f'N aceitou: {df5["aceitou"].sum():,}  |  N nao aceitou: {(df5["aceitou"]==0).sum():,}')

In [ ]:
# PIPELINE COMPLETO DO PROJETO
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# STEP 1: Features e alvo
ID5  = 'cnpj'
ALV5 = 'aceitou'
feats5 = [c for c in df5.columns if c not in [ID5, ALV5]]

NUMS5 = ['tempo_conta','vol_mensal','n_produtos','score_pj',
          'ja_tem_credito','inadim_hist']
CATS5 = ['setor']

# STEP 2: Feature engineering
df5['log_vol'] = np.log1p(df5['vol_mensal'])
NUMS5 = ['tempo_conta','log_vol','n_produtos','score_pj',
          'ja_tem_credito','inadim_hist']

X5 = df5[NUMS5 + CATS5]
y5 = df5[ALV5]

# STEP 3: Split
X5_tr, X5_te, y5_tr, y5_te = train_test_split(
    X5, y5, test_size=0.20, random_state=42, stratify=y5
)

# STEP 4: Preprocessador
pre5 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), NUMS5),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore',
                                            sparse_output=False))]), CATS5)
])

# STEP 5: Modelo (comecar pela logistica)
modelo5 = Pipeline([
    ('pre', pre5),
    ('clf', LogisticRegression(C=0.5, max_iter=1000, random_state=42))
])

# STEP 6: CV para estimar performance
kf5 = KFold(5, shuffle=False)
res5 = cross_validate(modelo5, X5_tr, y5_tr, cv=kf5,
                       scoring='roc_auc', return_train_score=True)
print(f'AUC CV: {res5["test_score"].mean():.4f} +/- {res5["test_score"].std():.4f}')

# STEP 7: Treinar no treino completo e avaliar no teste
modelo5.fit(X5_tr, y5_tr)
prob5_te = modelo5.predict_proba(X5_te)[:,1]
auc5     = roc_auc_score(y5_te, prob5_te)
print(f'AUC teste: {auc5:.4f}')

# STEP 8: Calcular o LIFT (metrica de negocio)
df_lift5 = pd.DataFrame({
    'score': prob5_te,
    'aceitou': y5_te.values
}).sort_values('score', ascending=False).reset_index(drop=True)

taxa_global5 = df_lift5['aceitou'].mean()
print()
print('=== CURVA DE LIFT ===')
print(f'{"% abordados":>15} {"Taxa no grupo":>15} {"Lift":>10}')
print('-' * 45)
for pct in [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
    nk = max(1, int(len(df_lift5) * pct))
    tk = df_lift5['aceitou'][:nk].mean()
    lift = tk / taxa_global5
    print(f'{pct*100:>14.0f}%  {tk*100:>14.1f}%  {lift:>10.2f}x')

In [ ]:
# STEP 9: SCORING DA CARTEIRA E ENTREGA PARA O NEGOCIO

# Rodando o modelo em TODA a carteira (nao so no teste)
df5['score_propensao'] = modelo5.predict_proba(X5[NUMS5 + CATS5])[:,1]

# Segmentando por faixa de propensao
df5['faixa'] = pd.cut(
    df5['score_propensao'],
    bins=[0, 0.05, 0.10, 0.20, 0.40, 1.0],
    labels=['Muito baixa','Baixa','Media','Alta','Muito alta']
)

# IMPACTO FINANCEIRO DA CAMPANHA
custo_campanha = 20.0   # R$ por abordagem
receita_contrato = 5000.0  # R$ de receita por contrato fechado

resumo_negocio = df5.groupby('faixa').agg(
    n_clientes=('cnpj','count'),
    taxa_aceite_media=('aceitou','mean'),
    score_medio=('score_propensao','mean')
).round(3)

resumo_negocio['custo_total'] = (resumo_negocio['n_clientes'] * custo_campanha).astype(int)
resumo_negocio['contratos_esperados'] = (
    resumo_negocio['n_clientes'] * resumo_negocio['taxa_aceite_media']
).round(0).astype(int)
resumo_negocio['receita_esperada'] = (
    resumo_negocio['contratos_esperados'] * receita_contrato
).astype(int)
resumo_negocio['roi'] = (
    resumo_negocio['receita_esperada'] / resumo_negocio['custo_total']
).round(1)

print('=== RESUMO PARA O GESTOR ===')
print()
print(resumo_negocio.to_string())
print()

# Top 10% mais propensos
top10 = df5.nlargest(int(N5*0.10), 'score_propensao')
taxa_top10 = top10['aceitou'].mean()
lift_top10 = taxa_top10 / taxa_global5

print('=== RECOMENDACAO EXECUTIVA ===')
print()
print(f'  Total da carteira: {N5:,} clientes')
print(f'  Taxa de aceite global: {taxa_global5*100:.1f}%')
print()
print(f'  Estrategia recomendada: abordar apenas os {int(N5*0.10):,} mais propensos')
print(f'  Custo da campanha focada:     R${int(N5*0.10)*custo_campanha:,.0f}')
print(f'  Taxa de aceite esperada:      {taxa_top10*100:.1f}%  ({lift_top10:.1f}x o aleatorio)')
print(f'  Contratos esperados:          {int(int(N5*0.10)*taxa_top10):,}')
print()
print(f'  VS campanha para todos:')
print(f'  Custo:     R${N5*custo_campanha:,.0f}')
print(f'  Contratos: {int(N5*taxa_global5):,}')
print()
print('MENSAGEM PARA O DIRETOR:')
print('  Abordando os 10% mais propensos:')
print(f'  -> Custo 10x menor | Contratos {lift_top10:.1f}x acima do aleatorio')
print(f'  -> ROI estimado: {(int(N5*0.10)*taxa_top10*receita_contrato)/(int(N5*0.10)*custo_campanha):.1f}x o investimento')

---
# CAPITULO 6 -- Checklists e Armadilhas
## O que separa o iniciante do profissional

---

## Checklist antes de entregar qualquer modelo

**Sobre o dado:**
- [ ] Removi o ID e qualquer coluna identificadora do dado de treino?
- [ ] Ha data leakage? Alguma feature so existe depois do evento?
- [ ] Tratei os ausentes com a mediana do treino (nao do dado todo)?
- [ ] Criei indicador binario para ausentes sistematicos?
- [ ] O split foi feito ANTES do preprocessamento?
- [ ] Usei stratify=y para classificacao desbalanceada?

**Sobre o modelo:**
- [ ] Tenho um baseline idiota para comparar?
- [ ] Usei Cross-Validation para comparar modelos (nao o conjunto de teste)?
- [ ] A metrica do CV usa `neg_*`? Multipliquei por -1?
- [ ] O conjunto de teste foi usado apenas UMA VEZ, no final?
- [ ] Os resultados fazem sentido de negocio?

**Sobre a avaliacao:**
- [ ] Se o alvo e desbalanceado, evitei usar acuracia?
- [ ] Reportei AUC + Lift, nao so AUC?
- [ ] Calculei o impacto em R$ ou % para o gestor?
- [ ] A curva ROC esta acima da diagonal? (se nao: algo errado)
- [ ] A feature importance faz sentido de negocio?

**Sobre a producao:**
- [ ] O modelo sera retreinado periodicamente?
- [ ] Ha monitoramento de drift (PSI)?
- [ ] Se for real-time: qual a latencia esperada?
- [ ] O modelo foi versionado (MLflow)?

---

## As 10 armadilhas mais comuns

1. **Usar o dado de teste para escolher hiperparametros** -> contaminacao
2. **Nao usar stratify em dados desbalanceados** -> split com proporcao errada
3. **Esquecer de multiplicar neg_* por -1** -> metrica negativa relatada
4. **Incluir o ID no modelo** -> o modelo memoriza IDs, nao generaliza
5. **Normalizar antes do split** -> data leakage via media/std
6. **Usar acuracia em dado desbalanceado** -> resultado enganoso
7. **Confundir R^2 com R^2 ajustado** -> R^2 nunca diminui, o ajustado pode
8. **MAPE com zeros no alvo** -> divisao por zero, resultado indefinido
9. **Nao inspecionar feature importance** -> pode revelar leakage
10. **Entregar AUC sem contexto de negocio** -> gestor nao entende AUC

---

## Dicionario de situacoes -> solucoes

| Situacao | O que fazer |
|----------|-------------|
| AUC treino muito maior que validacao | Overfitting: regularizar, simplificar |
| AUC treino e validacao ambos baixos | Underfitting: mais features, modelo complexo |
| AUC proxima de 1.0 (suspeita) | Data leakage: inspecionar features |
| Feature importance muito concentrada | Possivel leakage: investigar essa feature |
| Metrica negativa no cross_validate | Esqueceu de * -1 nos scoring com `neg_` |
| `KeyError` ao dar transform no teste | Categoria nova: `handle_unknown='ignore'` |
| Dataset muito lento de processar | Usar amostra para explorar, full para treinar |
| Dados temporais | Split por data, nao aleatorio; usar TimeSeriesSplit |
| Categorica com 100+ categorias | Target encoding ou embeddings, nao OHE |
| Modelo nao converge (sklearn warning) | Aumentar max_iter ou normalizar os dados |


In [ ]:
# EXERCICIO FINAL -- DIAGNOSTICO: O QUE ESTA ERRADO?

# [SITUACAO]
# Um colega entregou o codigo abaixo para voce revisar.
# Encontre TODOS os erros antes de executar.

print('=== CODIGO COM ERROS PARA DIAGNOSTICAR ===')
print()
codigo_errado = [
    '# ERRO 1: normalizar ANTES do split',
    'scaler = StandardScaler()',
    'X_norm = scaler.fit_transform(X)   # <-- leakage!',
    'X_tr, X_te = train_test_split(X_norm)',
    '',
    '# ERRO 2: ID no modelo',
    'X = df[["cnpj", "score", "faturamento"]]  # <-- cnpj e ID!',
    '',
    '# ERRO 3: usar o teste para escolher hiperparametros',
    'for C in [0.01, 0.1, 1.0]:',
    '    modelo = LogisticRegression(C=C)',
    '    modelo.fit(X_tr, y_tr)',
    '    auc = roc_auc_score(y_te, modelo.predict_proba(X_te)[:,1])  # <-- teste!',
    '',
    '# ERRO 4: neg_mean_squared_error nao multiplicado por -1',
    'res = cross_validate(modelo, X, y, scoring="neg_mean_squared_error")',
    'print("MSE:", res["test_score"].mean())   # <-- negativo!',
    '',
    '# ERRO 5: acuracia em dado desbalanceado',
    '# base tem 2% de fraudes',
    'print("Acuracia:", modelo.score(X_te, y_te))  # <-- inutil aqui!',
]
for linha in codigo_errado:
    print(f'  {linha}')

print()
print('=== CODIGO CORRETO ===')
print()
codigo_certo = [
    '# CORRETO 1: split primeiro, normalizar depois (dentro do Pipeline)',
    'X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y)',
    'pipe = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression())])',
    '',
    '# CORRETO 2: sem o ID',
    'X = df[["score", "faturamento"]]  # cnpj removido',
    '',
    '# CORRETO 3: usar CV (nao o teste) para escolher hiperparametros',
    'for C in [0.01, 0.1, 1.0]:',
    '    res = cross_validate(LogisticRegression(C=C), X_tr, y_tr, cv=5, scoring="roc_auc")',
    '    print(C, res["test_score"].mean())   # <-- CV, nao teste',
    '',
    '# CORRETO 4: * -1 para metricas neg_*',
    'res = cross_validate(modelo, X, y, scoring="neg_mean_squared_error")',
    'print("MSE:", -res["test_score"].mean())  # <-- * -1',
    '',
    '# CORRETO 5: AUC para dado desbalanceado',
    'print("AUC:", roc_auc_score(y_te, modelo.predict_proba(X_te)[:,1]))',
]
for linha in codigo_certo:
    print(f'  {linha}')

In [ ]:
# RESUMO FINAL: O MAPA MENTAL COMPLETO
print('=' * 70)
print('MAPA MENTAL: DA SITUACAO AO CODIGO')
print('=' * 70)
print()

mapa = [
    ('TIPO DE PROBLEMA', [
        ('Prever numero continuo',     'Regressao: LinearRegression, Ridge, XGBRegressor'),
        ('Prever categoria (0/1)',      'Classificacao: LogisticRegression, XGBClassifier'),
        ('Prever categoria (A/B/C)',    'Multiclasse: LogisticRegression(multi_class=ovr)'),
        ('Agrupar sem rotulo',          'Clustering: KMeans, hierarchical'),
        ('Reduzir dimensoes',           'PCA, UMAP'),
        ('Medir impacto causal',        'A/B test, DiD'),
    ]),
    ('METRICA CERTA', [
        ('Classificacao balanceada',    'Acuracia, F1, AUC'),
        ('Classificacao desbalanceada', 'AUC-ROC, Precision, Recall (NUNCA so acuracia)'),
        ('Regressao sem zeros',         'RMSE, MAE, R^2, MAPE'),
        ('Regressao com zeros no alvo', 'RMSE, MAE, R^2 (NUNCA MAPE)'),
        ('Negocio',                     'Lift, ROI, R$ economizado, % de captura'),
    ]),
    ('PREPROCESSAMENTO', [
        ('Numerico com NaN',            'SimpleImputer(strategy=median) + indicador binario'),
        ('Categorico com NaN',          'SimpleImputer(strategy=most_frequent)'),
        ('Categorico sem ordem',        'OneHotEncoder(handle_unknown=ignore)'),
        ('Categorico com ordem',        'OrdinalEncoder com ordem explicita'),
        ('Escala discrepante',          'StandardScaler (obrigatorio p/ SVM, Logistica)'),
        ('Distribuicao lognormal',      'log1p() antes de usar'),
    ]),
    ('ARMADILHAS', [
        ('neg_* no cross_validate',     'Sempre * -1: -res[test_score].mean()'),
        ('Normalizar antes do split',   'Sempre split primeiro -> Pipeline'),
        ('ID no modelo',               'Sempre remover ID/chave primaria'),
        ('Usar teste para tunar',       'Sempre CV -> teste so 1x no final'),
        ('Dado desbalanceado',          'Sempre stratify=y no split'),
    ]),
]

for categoria, itens in mapa:
    print(f'  {categoria}:')
    for situacao, solucao in itens:
        print(f'    [{situacao}]')
        print(f'     -> {solucao}')
    print()